In [37]:
import geopandas as gpd
import shapely
import os

In [32]:
input_folder = '../../scenarios/noumea_with_timetable/inputs/'
zonage_iris = gpd.read_file(input_folder + 'zonage_iris.geojson')
zonage_iris.reset_index(names='zone_id', inplace=True)

In [33]:
zonage_iris['geometry'] = zonage_iris['geometry'].centroid
zonage_iris = zonage_iris[['zone_id', 'nom', 'commune', 'quartier', 'geometry']]
od = zonage_iris.merge(zonage_iris, how='cross', suffixes=['_origin', '_destination'])

C:\Users\lrapin\AppData\Local\Temp\ipykernel_29056\2222716707.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zonage_iris['geometry'] = zonage_iris['geometry'].centroid


In [34]:
od['geometry'] = od.apply(lambda x: shapely.LineString([x['geometry_origin'], x['geometry_destination']]), 1)
od.drop(columns=['geometry_origin', 'geometry_destination'], inplace=True)

In [36]:
### Question: faudrait-il donner des matrices OD différentes en fonction de l'heure de la journée?
od['volume'] = 0
od.loc[(od['nom_origin'] == 'centre_ville') & (od['nom_destination'] == 'dumbea_sur_mer'), 'volume'] = 2500
od.loc[(od['nom_destination'] == 'centre_ville') & (od['nom_origin'] == 'dumbea_sur_mer'), 'volume'] = 5000

In [41]:
try:
    os.makedirs(input_folder + 'od')
except FileExistsError:
    pass

od.set_crs(epsg=4326, inplace=True)
od.to_file(input_folder + 'od/od.geojson')